这篇文档讲的是如何把一个不到 3 亿参数的“蚊子级”模型微调成一个表情符号（Emoji）翻译器，并塞进浏览器里跑 。虽然这个模型规模对我们搞大集群、跑千万核的人来说就像个玩具，但作为你入门 LLM 的“复健练习”倒是正合适。

# 第一阶段：环境初始化

Colab 默认环境比较杂乱。作为红帽专家，我要求你第一步先确认你的“工位”是否整洁。我们需要的是 T4 GPU 。

In [2]:
!nvidia-smi -L

GPU 0: NVIDIA L4 (UUID: GPU-3088a122-edd2-6bfe-6126-ddbf3b6fcb1f)


In [ ]:
from google.colab import drive

# 强制挂载 Google Drive 到 /content/drive 节点
# force_remount=True 可以防止之前挂载的残留导致异常
drive.mount('/content/drive', force_remount=True)


In [ ]:
import os

# 定义你的专属工作目录，例如 gemma3_finetune
gdrive_work_dir = '/content/drive/MyDrive/colab/2026/qwen3_finetune'

# 如果目录不存在，使用 os 模块自动创建
os.makedirs(gdrive_work_dir, exist_ok=True)

print(f"工作目录已准备就绪: {gdrive_work_dir}")

# 可以用 bash 命令验证目录创建成功
!ls -ld {gdrive_work_dir}


In [ ]:
%%bash
set -e

cd /content

if [ -f "/content/qwen3-emoji-full.tar.gz" ]; then
    echo "Found local tar package, skipping copy."
else
    echo "Copying from Google Drive..."
    cp {work_dir}/qwen3-emoji-full.tar.gz /content/

    tar -xzf /content/qwen3-emoji-full.tar.gz -C /
fi

ls -lh /content/qwen3-emoji-full/


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # 可写可不写；你之前看到默认就是 bf16
    device_map="auto",
)

print("vocab_size:", model.config.vocab_size)
print("pad_token:", tokenizer.pad_token)
print("param dtype:", next(model.parameters()).dtype)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

vocab_size: 151936
pad_token: <|endoftext|>
param dtype: torch.bfloat16


In [ ]:
from datasets import Dataset

raw_data = [
    {"input": "what a fun party", "output": "🥳🎉"},
    {"input": "good morning", "output": "☀️😎"},
]

raw_dataset = Dataset.from_list(raw_data)

def format_prompt(example):
    return {
        "text": f"User: Translate to emoji: {example['input']}\nAssistant: {example['output']}"
    }

formatted_dataset = raw_dataset.map(format_prompt)

def tokenize_fn(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length", # ✅ 打开 padding
        max_length=64, # emoji 任务很短，64 足够
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = formatted_dataset.map(
    tokenize_fn,
    remove_columns=formatted_dataset.column_names
)

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="./qwen3-emoji",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-5,
    fp16=False,
    bf16=True, #
    logging_steps=10,
    save_steps=500,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,  # 你的数据
    data_collator=data_collator,
)

print("--- 开始训练 ---")
trainer.train()
print("--- 训练完成 ---")


In [ ]:
def test_model(text):
    prompt = (
        "User: Translate to emoji: " + text + "\n"
        "Assistant: <|thinking|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.6,     # 推理模式推荐 0.6
        top_p=0.95,          # 配合采样，增加多样性
        repetition_penalty=1.1, # 增加重复惩罚，防止 </s> 连吐
        eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")],
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=False))


In [ ]:
# def test_model(text):
#     prompt = f"User: Translate to emoji: {text}\nAssistant: "
#     inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
#     outputs = model.generate(**inputs, max_new_tokens=10)
#     print(tokenizer.decode(outputs[0], skip_special_tokens=True))

test_model("working late at night")

In [ ]:
# for lora only

# # 将补丁缝合回原模型
# merged_model = model.merge_and_unload()

# # 保存这个完整的模型
# merged_model.save_pretrained("./gemma-3-emoji-full")
# tokenizer.save_pretrained("./gemma-3-emoji-full")

# print("模型已缝合，准备进入压缩环节。")

In [ ]:
trainer.save_model("./qwen3-emoji-full")
tokenizer.save_pretrained("./qwen3-emoji-full")


In [ ]:
# 训练完成后，打包备份到 2T 的 Google Drive 当中
!tar -czf {gdrive_work_dir}/qwen3-emoji-full.tar.gz /content/qwen3-emoji-full/


In [ ]:
!nvidia-smi

In [ ]:
# # 释放 GPU
# del trainer
# del model
# del data_collator

# import torch
# torch.cuda.empty_cache()

# import gc
# gc.collect()

# !nvidia-smi

# print("GPU 已释放，可以继续下一步。")


In [ ]:
%%bash
set -e

# 如果还没 clone，就 clone
if [ ! -d "llama.cpp" ]; then
  git clone https://github.com/ggerganov/llama.cpp
fi

cd llama.cpp

# 创建 build 目录
mkdir -p build
cd build

# 配置（CPU 版本即可，转换不需要 GPU）
cmake ..

# 编译
cmake --build . -j


In [ ]:
# 训练完成后，打包备份到 2T 的 Google Drive 当中
!tar -czf {gdrive_work_dir}/llama.cpp.tar.gz /content/llama.cpp/


In [ ]:
%%bash
set -e
cd llama.cpp

python convert_hf_to_gguf.py \
  /content/qwen3-emoji-full \
  --outfile /content/qwen3-0.6b-bf16.gguf \
  --outtype bf16


In [ ]:
%%bash
set -e
cd llama.cpp/build/bin

./llama-quantize \
  /content/qwen3-0.6b-bf16.gguf \
  /content/qwen3-0.6b-q4.gguf \
  Q4_K_M


In [ ]:
# 训练完成后，打包备份到 2T 的 Google Drive 当中
# !tar -czf {gdrive_work_dir}/llama.cpp.tar.gz /content/llama.cpp/
!cp -f {gdrive_work_dir}/qwen3-0.6b-bf16.gguf /content/qwen3-0.6b-bf16.gguf
!cp -f {gdrive_work_dir}/qwen3-0.6b-q4.gguf /content/qwen3-0.6b-q4.gguf


In [ ]:
# from google.colab import files
# files.download("/content/qwen3-0.6b-q4.gguf")

In [ ]:
# 1. 安装 cloudflared
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb

In [ ]:


# 2. 启动你的 llama-server (假设端口是 8080)
# 注意：加 & 让它在后台运行
!/content/llama.cpp/build/bin/llama-server -m /content/qwen3-0.6b-q4.gguf --host 0.0.0.0 --port 18080 > server.log 2>&1 &

# 3. 启动隧道
!cloudflared tunnel --url http://localhost:18080